# Employee Data Analysis

Analyzing a sample employee dataset: cleaning, filtering, grouping, and deriving insights.

## 1. Load and Inspect

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

df = pd.read_csv("data/employees.csv")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)
print(f"\nMissing values:\n{df.isnull().sum()}")

## 2. Cleaning

We have missing values in Salary, PerformanceScore, and Department. Also some invalid dates and empty names.

In [ ]:
# Remove empty names
df = df[df["Name"].str.strip() != ""]

# Parse dates, coerce errors to NaT
df["JoinDate"] = pd.to_datetime(df["JoinDate"], errors="coerce")

# Fill missing Department with mode
df["Department"] = df["Department"].fillna(df["Department"].mode()[0])

# Fill missing numeric with median
for col in ["Salary", "PerformanceScore"]:
    df[col] = df[col].fillna(df[col].median())

# Cap salary outliers (beyond 3 std)
mean_sal, std_sal = df["Salary"].mean(), df["Salary"].std()
df["Salary"] = df["Salary"].clip(upper=mean_sal + 3 * std_sal)

# Filter unreasonable ages
df = df[(df["Age"] >= 18) & (df["Age"] <= 65)]

print(f"Cleaned dataset: {len(df)} rows")
print(f"Missing values now:\n{df.isnull().sum()}")

## 3. Filtering

In [ ]:
high_performers = df[df["PerformanceScore"] >= 90]
engineers = df[df["Department"] == "Engineering"]
senior_staff = df[(df["YearsExperience"] >= 10) & (df["Salary"] > 80000)]

print(f"High performers (score >= 90): {len(high_performers)}")
print(f"Engineering dept: {len(engineers)}")
print(f"Experienced (>10yr) earning >$80k: {len(senior_staff)}")

## 4. Grouping and Aggregation

In [ ]:
dept_stats = df.groupby("Department").agg(
    EmployeeCount=("EmployeeID", "count"),
    AvgSalary=("Salary", "mean"),
    AvgPerformance=("PerformanceScore", "mean"),
    AvgExperience=("YearsExperience", "mean"),
    AvgAge=("Age", "mean"),
).round(1).sort_values("AvgSalary", ascending=False)

dept_stats

In [ ]:
city_stats = df.groupby("City")["Salary"].agg(["count", "mean"]).round(1).sort_values("mean", ascending=False)
city_stats.head(10)

## 5. Key Insights

In [ ]:
best_dept = dept_stats["AvgPerformance"].idxmax()
highest_paid = dept_stats["AvgSalary"].idxmax()
corr_sal_perf = df["Salary"].corr(df["PerformanceScore"])
corr_exp_perf = df["YearsExperience"].corr(df["PerformanceScore"])
avg_tenure = (pd.Timestamp.now() - df["JoinDate"]).dt.days.mean() / 365.25
top_city = city_stats["mean"].idxmax()

print(f"1. Best performing dept: {best_dept} ({dept_stats.loc[best_dept, 'AvgPerformance']} avg)")
print(f"2. Highest paid dept: {highest_paid} (${dept_stats.loc[highest_paid, 'AvgSalary']:,.0f} avg)")
print(f"3. Salary vs Performance correlation: {corr_sal_perf:.3f}")
print(f"4. Experience vs Performance correlation: {corr_exp_perf:.3f}")
print(f"5. Average employee tenure: {avg_tenure:.1f} years")
print(f"6. Highest avg salary by city: {top_city} (${city_stats.loc[top_city, 'mean']:,.0f})")

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df["Salary"], bins=25, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Salary Distribution")

sns.boxplot(data=df, x="Department", y="PerformanceScore", ax=axes[0, 1])
axes[0, 1].set_title("Performance Score by Department")
axes[0, 1].tick_params(axis="x", rotation=45)

dept_stats["AvgSalary"].plot(kind="bar", ax=axes[1, 0], color="steelblue")
axes[1, 0].set_title("Average Salary by Department")
axes[1, 0].set_ylabel("Avg Salary ($)")
axes[1, 0].tick_params(axis="x", rotation=45)

sns.scatterplot(data=df, x="YearsExperience", y="PerformanceScore", hue="Department", alpha=0.6, ax=axes[1, 1])
axes[1, 1].set_title("Experience vs Performance")
axes[1, 1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("data/analysis_plots.png", dpi=150, bbox_inches="tight")
plt.show()